In [1]:
library('ape')
library('ggtree')
library('phytools')
library('ggplot2')

ggtree v3.1.4  For help: https://yulab-smu.top/treedata-book/

If you use ggtree in published research, please cite the most appropriate paper(s):

1. Guangchuang Yu. Using ggtree to visualize data on tree-like structures. Current Protocols in Bioinformatics, 2020, 69:e96. doi:10.1002/cpbi.96
2. Guangchuang Yu, Tommy Tsan-Yuk Lam, Huachen Zhu, Yi Guan. Two methods for mapping and visualizing associated data on phylogeny using ggtree. Molecular Biology and Evolution 2018, 35(12):3041-3043. doi:10.1093/molbev/msy194
3. Guangchuang Yu, David Smith, Huachen Zhu, Yi Guan, Tommy Tsan-Yuk Lam. ggtree: an R package for visualization and annotation of phylogenetic trees with their covariates and other associated data. Methods in Ecology and Evolution 2017, 8(1):28-36. doi:10.1111/2041-210X.12628




Attaching package: ‘ggtree’


The following object is masked from ‘package:ape’:

    rotate


Loading required package: maps



# The cell below is the production code to use for reconstructing discrete traits for insertions

Tips are associated with a vector of 'y' and 'n' to describe the presence/absence of a specific insertion, which have been identified a priori in the insertion processing notebook. These vectors are extracted from a csv that is processed in separate python code. Each insertion is reconstructed as a discrete trait in each tree that is called in the 'trees' vector. The output of this is then ready in the ASRtools python code for post-processing ancestral nodes to either include or disclude the specific insertions, depending on the posterior probability of that node including that insertion. 

In [2]:
x <- read.csv('Insertion_tip_dataframe.csv')
x$X <- NULL
tips <- x[['id']]

trees <- c('List of nwk tree files to process')

ins <- c(colnames(x))
insertions <- ins[-1]


for (n in trees){
    anc_tree_bl <- read.tree(paste(n,'anc_bl.nwk', sep = '_'))
    anc_tree <- read.tree(paste(n,'anc.nwk', sep = '_'))
    anc_tree$edge.length <- anc_tree_bl$edge.length
    anc_tree <- phytools::midpoint.root(anc_tree)
    node <- anc_tree$node.label
    d <- data.frame(node)
    
    tree <- read.tree(n)
    tree <- phytools::midpoint.root(tree)    
    for (i in insertions){
        ins <- x[[i]]
        names(ins) <- tips
        fitER <- ace(ins,tree, model = 'ER', type='discrete')
        fitER_df <- round(fitER$lik.anc,3)
        n_values <- fitER_df[,'n']
        d[i] <- n_values
    }
    lab <- paste(n,'insertion_mapping.csv', sep = '_')
    write.csv(d, lab)
}